In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
!pip install -q -U transformers peft datasets accelerate scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [3]:
import transformers, peft, datasets
print(transformers.__version__, peft.__version__, datasets.__version__)

5.23.0 0.21.3 5.2.0


In [10]:
import json, random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"   # pode trocar por outro modelo pequeno com PT-BR
N_TRAIN, N_TEST = 6000, 600                 # amostras balanceadas por classe
LABELS = ["negativo", "neutro", "positivo"]

In [12]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(repo_id="vladjr/B2W-Reviews01", filename="B2W-Reviews01.csv", repo_type="dataset")
raw = pd.read_csv(path)
print(raw.shape)
print(raw.columns.tolist())  

df = raw[["review_text", "overall_rating"]].dropna()
df = df[df.review_text.str.len().between(20, 600)]
df["label"] = df.overall_rating.astype(int).map(
    lambda r: "negativo" if r <= 2 else ("neutro" if r == 3 else "positivo"))

def sample_balanced(frame, n_total):
    per_class = n_total // len(LABELS)
    parts = [frame[frame.label == lab].sample(per_class, random_state=SEED) for lab in LABELS]
    return pd.concat(parts).sample(frac=1, random_state=SEED)

test_df = sample_balanced(df, N_TEST)                  # teste fixo, nunca usado no treino
train_df = sample_balanced(df.drop(test_df.index), N_TRAIN)
print(train_df.label.value_counts(), test_df.label.value_counts())

B2W-Reviews01.csv: reconstructing file:   0%|          |  0.00B / 49.5MB            

B2W-Reviews01.csv: downloading bytes:           |  0.00B            

(132373, 14)
['submission_date', 'reviewer_id', 'product_id', 'product_name', 'product_brand', 'site_category_lv1', 'site_category_lv2', 'review_title', 'overall_rating', 'recommend_to_a_friend', 'review_text', 'reviewer_birth_year', 'reviewer_gender', 'reviewer_state']
label
negativo    2000
neutro      2000
positivo    2000
Name: count, dtype: int64 label
negativo    200
positivo    200
neutro      200
Name: count, dtype: int64


In [13]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

SYSTEM = ("Você classifica a polaridade de avaliações de clientes em português. "
          "Responda apenas com uma palavra: positivo, neutro ou negativo.")

def build_prompt(text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def parse(output):
    out = output.strip().lower()
    for lab in LABELS:
        if out.startswith(lab):
            return lab
    return "invalido"

@torch.no_grad()
def predict(model, texts, bs=32):
    model.eval()
    tok.padding_side = "left"
    preds = []
    for i in range(0, len(texts), bs):
        batch = [build_prompt(t) for t in texts[i:i + bs]]
        enc = tok(batch, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        out = model.generate(**enc, max_new_tokens=4, do_sample=False, pad_token_id=tok.pad_token_id)
        preds += [parse(x) for x in tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
    tok.padding_side = "right"
    return preds

def evaluate(model, name):
    preds = predict(model, test_df.review_text.tolist())
    y = test_df.label.tolist()
    res = {"modelo": name,
           "accuracy": accuracy_score(y, preds),
           "macro_f1": f1_score(y, preds, average="macro", labels=LABELS),
           "invalidos": sum(p == "invalido" for p in preds)}
    print(res)
    print(classification_report(y, preds, labels=LABELS, zero_division=0))
    print(confusion_matrix(y, preds, labels=LABELS))
    return res

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [14]:
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to("cuda")
res_base = evaluate(model, "base (zero-shot)")

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'modelo': 'base (zero-shot)', 'accuracy': 0.615, 'macro_f1': 0.4923576818657945, 'invalidos': 1}
              precision    recall  f1-score   support

    negativo       0.64      0.96      0.77       200
      neutro       0.00      0.00      0.00       200
    positivo       0.59      0.88      0.71       200

   micro avg       0.62      0.61      0.62       600
   macro avg       0.41      0.61      0.49       600
weighted avg       0.41      0.61      0.49       600

[[193   0   6]
 [ 86   0 114]
 [ 24   0 176]]


In [19]:
def encode(row):
    prompt_ids = tok(build_prompt(row["review_text"]), add_special_tokens=False).input_ids
    answer_ids = tok(row["label"] + tok.eos_token, add_special_tokens=False).input_ids
    return {"input_ids": prompt_ids + answer_ids,
            "attention_mask": [1] * (len(prompt_ids) + len(answer_ids)),
            "labels": [-100] * len(prompt_ids) + answer_ids}

train_ds = Dataset.from_pandas(train_df[["review_text", "label"]].reset_index(drop=True))
train_ds = train_ds.map(encode, remove_columns=train_ds.column_names)

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [17]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [18]:
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


In [25]:
args = TrainingArguments(output_dir="out", per_device_train_batch_size=2,
                         gradient_accumulation_steps=8, learning_rate=2e-4,
                         num_train_epochs=1, lr_scheduler_type="cosine", warmup_steps=10,
                         logging_steps=25, save_strategy="no", fp16=True,
                         report_to="none", seed=SEED)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
trainer.train()

Step,Training Loss
25,0.155256
50,0.172767
75,0.210082
100,0.172448
125,0.166846
150,0.177559
175,0.166297


TrainOutput(global_step=188, training_loss=0.1757290908630858, metrics={'train_runtime': 1014.482, 'train_samples_per_second': 5.914, 'train_steps_per_second': 0.185, 'total_flos': 1561783095444480.0, 'train_loss': 0.1757290908630858, 'epoch': 1.0})

In [26]:
model.save_pretrained("/kaggle/working/adapter_polaridade")
tok.save_pretrained("/kaggle/working/adapter_polaridade")

('/kaggle/working/adapter_polaridade/tokenizer_config.json',
 '/kaggle/working/adapter_polaridade/chat_template.jinja',
 '/kaggle/working/adapter_polaridade/tokenizer.json')

In [27]:
res_ft = evaluate(model, "fine-tuned (LoRA)")
json.dump([res_base, res_ft], open("resultados.json", "w"), indent=2, ensure_ascii=False)

{'modelo': 'fine-tuned (LoRA)', 'accuracy': 0.765, 'macro_f1': 0.7636889979722475, 'invalidos': 0}
              precision    recall  f1-score   support

    negativo       0.84      0.90      0.86       200
      neutro       0.66      0.66      0.66       200
    positivo       0.79      0.74      0.77       200

    accuracy                           0.77       600
   macro avg       0.76      0.77      0.76       600
weighted avg       0.76      0.77      0.76       600

[[179  20   1]
 [ 31 131  38]
 [  4  47 149]]


In [30]:
import shutil
shutil.make_archive("/kaggle/working/adapter_polaridade", "zip", "/kaggle/working/adapter_polaridade")

'/kaggle/working/adapter_polaridade.zip'